# Colab driver notebook for `scripts/run_download.py`

Notebook này dùng để chạy bước download theo workflow:

1. mount Google Drive / Shared Drive
2. đi vào repo
3. pull code mới nhất từ GitHub
4. cài dependencies
5. gọi `scripts/run_download.py`
6. xem output:
   - `normalized_manifest.csv`
   - `download_log.csv`
   - thư mục `videos/`
   - thư mục `info_json/`

Notebook này **không chứa logic download chính**.  
Logic thật nằm trong:

- `src/ingest/excel_manifest.py`
- `src/ingest/youtube_download.py`
- `scripts/run_download.py`


## Expected repo structure

```text
repo/
├── requirements.txt
├── scripts/
│   └── run_download.py
└── src/
    └── ingest/
        ├── excel_manifest.py
        └── youtube_download.py
```


## Configure paths

In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount("/content/drive")

In [ ]:
from pathlib import Path

SHARED_DRIVE_ROOT = Path("/content/drive/Shareddrives/NLP4B/MAIN")
REPO_ROOT = SHARED_DRIVE_ROOT / "CODE"
GITHUB_REPO_URL = "https://github.com/CallmeAndree/NLP4B.git"
GIT_BRANCH = "main"

# ===== Input / output =====
INPUT_EXCEL_PATH = SHARED_DRIVE_ROOT / "manifests" / "link_videos.xlsx"
OUTPUT_ROOT = SHARED_DRIVE_ROOT / "downloaded_videos"

# ===== Download options =====
SHEET_NAME = 0
MAX_DOWNLOADS = 2         # đổi thành None khi chạy full
DOWNLOAD_MODE = "720p_mp4"
SLEEP_BETWEEN_DOWNLOADS = 1.0
OVERWRITE_EXISTING = False
WRITE_INFO_JSON = True
DEDUPLICATE_VIDEO_IDS = False

print("SHARED_DRIVE_ROOT =", SHARED_DRIVE_ROOT)
print("REPO_ROOT         =", REPO_ROOT)
print("INPUT_EXCEL_PATH  =", INPUT_EXCEL_PATH)
print("OUTPUT_ROOT       =", OUTPUT_ROOT)


## Prepare repo

Cell này sẽ:
- clone repo nếu chưa có
- hoặc `git pull` nếu repo đã tồn tại


In [ ]:
import subprocess

def run_cmd(cmd, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)

git_dir = REPO_ROOT / ".git"

if git_dir.exists():
    print("Repo exists. Pulling latest code...")
    run_cmd(["git", "fetch", "origin"], cwd=REPO_ROOT)
    run_cmd(["git", "checkout", GIT_BRANCH], cwd=REPO_ROOT)
    run_cmd(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_ROOT)
else:
    print("Repo does not exist. Cloning...")
    run_cmd(["git", "clone", "-b", GIT_BRANCH, GITHUB_REPO_URL, str(REPO_ROOT)])


## Install dependencies


In [ ]:
requirements_path = REPO_ROOT / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt not found: {requirements_path}")

run_cmd(["python", "-m", "pip", "install", "-r", str(requirements_path)])


## Validate input files


In [ ]:
if not INPUT_EXCEL_PATH.exists():
    raise FileNotFoundError(f"Input Excel file not found: {INPUT_EXCEL_PATH}")

script_path = REPO_ROOT / "scripts" / "run_download.py"
if not script_path.exists():
    raise FileNotFoundError(f"run_download.py not found: {script_path}")

print("Input Excel found:", INPUT_EXCEL_PATH)
print("Script found     :", script_path)


## Run the download script

Cell này sẽ:
- tạo `normalized_manifest.csv`
- tải video vào `videos/`
- lưu `.info.json` vào `info_json/`
- sinh `download_log.csv`
- sinh `download_archive.txt`


In [ ]:
run_cmd(["python", str(script_path), "--help"], cwd=REPO_ROOT)

In [ ]:
cmd = [
    "python",
    str(script_path),
    "--input-excel", str(INPUT_EXCEL_PATH),
    "--output-root", str(OUTPUT_ROOT),
    "--sheet-name", str(SHEET_NAME),
    "--download-mode", DOWNLOAD_MODE,
    "--sleep-between-downloads", str(SLEEP_BETWEEN_DOWNLOADS),
]

if MAX_DOWNLOADS is not None:
    cmd += ["--max-downloads", str(MAX_DOWNLOADS)]

if OVERWRITE_EXISTING:
    cmd += ["--overwrite-existing"]

if not WRITE_INFO_JSON:
    cmd += ["--no-info-json"]

if DEDUPLICATE_VIDEO_IDS:
    cmd += ["--deduplicate-video-ids"]

run_cmd(cmd, cwd=REPO_ROOT)


## Inspect outputs


In [ ]:
normalized_manifest_path = OUTPUT_ROOT / "normalized_manifest.csv"
download_log_path = OUTPUT_ROOT / "download_log.csv"
videos_dir = OUTPUT_ROOT / "videos"
info_json_dir = OUTPUT_ROOT / "info_json"

print("normalized_manifest.csv:", normalized_manifest_path)
print("download_log.csv       :", download_log_path)
print("videos_dir             :", videos_dir)
print("info_json_dir          :", info_json_dir)


In [ ]:
import pandas as pd

if normalized_manifest_path.exists():
    manifest_df = pd.read_csv(normalized_manifest_path)
    print("normalized_manifest.csv preview")
    display(manifest_df.head(20))
else:
    print("normalized_manifest.csv not found")

if download_log_path.exists():
    log_df = pd.read_csv(download_log_path)
    print("download_log.csv preview")
    display(log_df.head(20))

    summary = {
        "total_rows": len(log_df),
        "ok": int((log_df["status"] == "ok").sum()) if "status" in log_df.columns else None,
        "error": int((log_df["status"] == "error").sum()) if "status" in log_df.columns else None,
    }
    print(summary)
else:
    print("download_log.csv not found")


In [ ]:
print("Videos:")
if videos_dir.exists():
    for p in sorted(videos_dir.glob("*"))[:20]:
        print("-", p.name)
else:
    print("videos/ not found")

print()
print("Info JSON files:")
if info_json_dir.exists():
    for p in sorted(info_json_dir.glob("*"))[:20]:
        print("-", p.name)
else:
    print("info_json/ not found")


## Recommended usage

### First run
- đặt `MAX_DOWNLOADS = 2`
- kiểm tra:
  - `video_id`
  - `title`
  - `normalized_manifest.csv`
  - `download_log.csv`

### Full run
- đổi `MAX_DOWNLOADS = None`
- chạy lại cell download script


## Next step

Sau khi bước download ổn định, notebook tiếp theo nên là:

- `02_extract_video_metadata.ipynb`

Notebook đó sẽ đọc:
- `normalized_manifest.csv`
- `videos/`
- `info_json/`

và sinh:
- `video_metadata.csv`
- `metadata_log.csv`
